### Summarize the file

In [58]:
import os
import gradio as gr

from langchain_community.document_loaders import (
    Docx2txtLoader,
    PyPDFLoader,
    UnstructuredPowerPointLoader,
    UnstructuredExcelLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama


In [59]:
def load_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".txt":
        loader = Docx2txtLoader(file_path)
    elif ext == ".pdf":
        loader = PyPDFLoader(file_path)
    elif ext in [".ppt", ".pptx"]:
        loader = UnstructuredPowerPointLoader(file_path)
    elif ext in [".xls", ".xlsx"]:
        loader = UnstructuredExcelLoader(file_path)

    return loader.load()

In [60]:
def split_text(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200
    )
    return splitter.split_documents(docs)

In [77]:
def summarize_docs(chunks):
    llm = ChatOllama(model="llama3.1:latest")

    prompt = ChatPromptTemplate.from_template(
        "Summarize the following content clearly:\n\n{text}"
    )

    chain  = prompt | llm | StrOutputParser()

    partial_summarizes = []

    for doc in chunks:
        partial_summarizes.append(chain.invoke({"text": doc.page_content}))
        
    combined = ".\n\n".join(partial_summarizes)

    final_prompt = ChatPromptTemplate.from_template(
        "Create a clean final summary from the following:\n\n{text}"
    )


    final_chain = final_prompt | llm | StrOutputParser()

    full_summary = final_chain.invoke({"text":combined})

    return full_summary





In [78]:
def summarize_file(file):
    try:
        docs = load_file(file.name)
        chunks = split_text(docs)
        summary =summarize_docs(chunks)
        return summary
    except Exception as e:
        return f"Error : str{e}"

In [80]:
input_file = gr.File(label= "Upload your file")
selector = gr.Dropdown([".docs", ".pdf", ".ppt",".pptx", ".xls",".xlsx"], label = "Select file format", value=".pdf")
output = gr.Markdown(label = "Response")

view = gr.Interface(
    fn = summarize_file,
    inputs = [input_file],
    outputs = output,
    title="📄 AI File Summarizer (Ollama)",
    flagging_mode = "never"

)
if __name__ == "__main__":
    view.launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.
